In [1]:
import pandas as pd
import numpy as np

RAW = "s3://sagemaker-salesmesh-myra/raw/Base.csv"
OUT = "s3://sagemaker-salesmesh-myra/processed/"

CORE = ["name_email_similarity", "customer_age", "prev_address_months_count",
        "current_address_months_count", "date_of_birth_distinct_emails_4w",
        "email_is_free", "phone_home_valid", "phone_mobile_valid", "foreign_request",
        "source", "session_length_in_minutes", "device_os", "keep_alive_session",
        "device_distinct_emails_8w"]
SCALE = ["velocity_6h", "velocity_24h", "velocity_4w", "zip_count_4w"]
LABEL = "fraud_bool"

df = pd.read_csv(RAW, usecols=CORE + SCALE + [LABEL, "month"])
print(df.shape)

(1000000, 20)


In [2]:
NEG_IS_MISSING = ["prev_address_months_count", "current_address_months_count",
                  "session_length_in_minutes", "device_distinct_emails_8w", "velocity_6h"]

for col in NEG_IS_MISSING:
    df.loc[df[col] < 0, col] = np.nan

df.isna().mean().sort_values(ascending=False).head(6)

prev_address_months_count       0.712920
current_address_months_count    0.004254
session_length_in_minutes       0.002015
device_distinct_emails_8w       0.000359
velocity_6h                     0.000044
name_email_similarity           0.000000
dtype: float64

In [3]:
num = [c for c in CORE + SCALE if c not in ("source", "device_os")]
df.groupby(LABEL)[num].mean().T.round(3)

fraud_bool,0,1
name_email_similarity,0.495,0.393
customer_age,33.609,40.859
prev_address_months_count,60.649,83.552
current_address_months_count,86.649,114.948
date_of_birth_distinct_emails_4w,9.527,7.443
email_is_free,0.528,0.661
phone_home_valid,0.419,0.253
phone_mobile_valid,0.890,0.851
foreign_request,0.025,0.050
session_length_in_minutes,7.555,8.245


In [4]:
train = df[df["month"] <= 5].drop(columns="month")
test  = df[df["month"] >= 6].drop(columns="month")

print("Train:", train.shape, "fraud rate:", round(train[LABEL].mean() * 100, 3), "%")
print("Test: ", test.shape,  "fraud rate:", round(test[LABEL].mean() * 100, 3), "%")

Train: (794989, 19) fraud rate: 1.025 %
Test:  (205011, 19) fraud rate: 1.404 %


In [ ]:
train.to_csv(OUT + "train.csv", index=False)
test.to_csv(OUT + "test.csv", index=False)
print("Saved.")